# Visualising data 2: what does one data point show?

Last week we made bar charts, scatter plots and line plots. This week we build the kind of figures you see in
journal articles, and the figures you will make for your capstone and research projects.

All the way through we ask one question:

> **What does one point (or bar, line or shaded band) on this graph represent, and which measurement did it come from?**

Every mark on a figure summarises some measurements. A dot might be a single button press, or a person's average
over hundreds of button presses, or the average of a whole group. Readers often misread a graph because
they don't know which of these they are looking at. Authors often mislead, usually by accident, for the same reason.

We will climb a ladder, starting from the raw measurement and summarising a bit more at each step:

| Step | One point is... | Data |
|---|---|---|
| 1 | one **trial** (one button press) | flanker task |
| 2 | one **participant** (their average over trials) | flanker task |
| 3 | one participant's **effect** (a difference between conditions) | flanker task |
| 4 | one **group** mean, and what its error bar means | flanker task |
| 5 | the group's average signal at one **moment in time** | fMRI |
| 6 | one cell of a **multi-factor** design, in small multiples | monkey neurons |
| 7 | putting it together: a **figure for a paper** | flanker task |

All the data are real and published.

**Using this notebook later.** Sections headed **Reference** explain each new command and its most useful
arguments. You don't need to read them during the lecture; come back to them when you do the exercises.
There is a summary table of all the commands at the end.

## Setup

In [ ]:
# Store and reshape tables with pandas.
import pandas as pd
# Matplotlib gives us control over figures, axes and saving.
import matplotlib.pyplot as plt
# Seaborn draws statistical graphs from DataFrames.
import seaborn as sns

# "ticks" gives a clean white background with tick marks, closer to journal style than "whitegrid".
sns.set_theme(style="ticks", context="notebook")

We will use the same colour for each condition on **every** graph today. Then a reader who has learnt
"orange = incongruent" in one panel can carry that over to the next. These colours come from a colour-blind-safe palette
(Okabe & Ito). We also fix the **order** of the conditions once, so every graph uses the same order.

In [ ]:
# One colour per condition, reused on every graph.
# This is a dictionary: {key: value}. Passing it as palette= tells seaborn exactly which colour
# goes with which condition name, so the colours can't swap between graphs.
# The "#0072B2" strings are hex colour codes (red, green, blue in hexadecimal).
condition_colours = {
    "congruent": "#0072B2",    # blue
    "neutral": "#999999",      # grey
    "incongruent": "#D55E00",  # orange
}
# The order in which conditions should appear on axes and in legends.
condition_order = ["congruent", "neutral", "incongruent"]

## The experiment: the flanker task

On each trial a row of five symbols appears. The participant presses a key to report which way the **centre**
arrow points, ignoring the arrows on either side (the *flankers*).

| Condition | Example | Flankers... |
|---|---|---|
| congruent | `< < < < <` | point the same way as the centre arrow |
| incongruent | `> > < > >` | point the opposite way |
| neutral | `- - < - -` | don't point either way |

People are slower and make more errors when the flankers conflict with the target. This is the
**flanker effect**, a classic measure of attentional control.

Data: Hedge, Powell & Sumner (2018), *Behavior Research Methods*. 47 participants each did 720 trials in 5 blocks.
Very fast (< 200 ms) and very slow (> 2000 ms) responses were removed before the data were shared.

In [ ]:
# Each row of this file is one trial.
flanker_df = pd.read_csv("data/flanker_hedge2018.csv")

# Turn condition into an *ordered* category, so seaborn and pandas use our order everywhere.
# Without this, seaborn would order the conditions by whichever appears first in the file.
# pd.Categorical(values, categories=[...], ordered=True) keeps the same text but remembers the order.
flanker_df["condition"] = pd.Categorical(
    flanker_df["condition"], categories=condition_order, ordered=True
)

flanker_df.head()

**What does one row show?** One trial: one participant, one block, one press of a key. `rt_ms` is the time from
the arrows appearing to the key press, in milliseconds. `correct` is 1 for a correct response and 0 for an error.

In [ ]:
# How many trials and participants are there?
print("Trials:", len(flanker_df))
print("Participants:", flanker_df["participant"].nunique())
# Proportion of correct trials in each condition. correct is 0 or 1, so its mean is the proportion correct.
# observed=True is needed when grouping by a Categorical column: it tells pandas to use only the
# categories that actually occur (and avoids a warning).
flanker_df.groupby("condition", observed=True)["correct"].mean().round(3)

Reaction times are usually analysed for **correct trials only**, because an error is a different kind of
response. We keep a separate DataFrame for those trials.

In [ ]:
# Keep correct trials for the reaction-time analysis.
correct_df = flanker_df.loc[flanker_df["correct"] == 1]
len(correct_df)

## Start at the end: the figure you usually see

This is the kind of figure that often appears in a paper: mean reaction time in each condition, with error bars.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

# Seaborn averages every row in each condition and adds an error bar (by default a 95% CI).
sns.barplot(
    data=correct_df,
    x="condition",
    y="rt_ms",
    hue="condition",
    palette=condition_colours,
    # hue="condition" is the same column as x: we use it only to colour each bar.
    # dodge=False stops seaborn shifting the bars sideways to make room for hue groups.
    dodge=False,
    ax=ax,
)
ax.set(xlabel="", ylabel="Mean reaction time (ms)", title="Flanker task")
plt.show()

Before reading on, try to answer these from the graph alone:

1. How many people are in this study?
2. Did *everyone* respond more slowly on incongruent trials, or only some people?
3. What does the error bar show? What does the bar itself average over?

We can't answer any of these from the graph. To see why, we start at the bottom of the ladder.
We come back to question 3 in step 4.

## Step 1: one point = one trial

Look at the raw measurements for **one** participant first. Each dot below is one press of a key.
Note: run with jitter =0.0 and jitter =0.3 to demonstrate how a stripplot reveals the distribution. 

In [ ]:
# Select the correct trials for participant 1.
one_person_df = correct_df.loc[correct_df["participant"] == 1]

fig, ax = plt.subplots(figsize=(7, 3.5))
# A strip plot draws every observation as a dot.
# x is numeric and y is categorical, so seaborn draws the strips horizontally.
# jitter spreads the dots up and down a little, so they don't hide each other.
sns.stripplot(
    data=one_person_df,
    x="rt_ms",
    y="condition",
    hue="condition",
    palette=condition_colours,
    jitter=0.0,
    alpha=0.5,
    size=3,
    ax=ax,
)
ax.set(xlabel="Reaction time (ms)", ylabel="", title="Participant 1: one dot = one trial")
plt.show()

What do we see?

- The same person, doing the same thing, gives very different reaction times from one trial to the next:
  from about 250 ms to well over 1,000 ms.
- The incongruent dots sit only a little to the right. The shift (tens of ms) is **much smaller than the
  trial-to-trial spread** (hundreds of ms).
- If someone picked one trial at random and told you its RT, you couldn't say which condition it came from.

This is why experiments repeat trials many times: the average over many trials is much more stable than any single trial.

#### Reference: `sns.stripplot()`

Draws **one dot per row** of the DataFrame, in a strip for each category. Use it whenever you want readers to see
every observation.

```python
sns.stripplot(data=df, x="category_column", y="number_column", hue="category_column",
              palette=colour_dict, jitter=0.2, alpha=0.5, size=4, ax=ax)
```

| Argument | What it does |
|---|---|
| `x`, `y` | One categorical column and one numeric column. Whichever is numeric sets the direction (numeric on `x` → horizontal strips). |
| `hue`, `palette` | Colour the dots by a column; `palette` can be a dict of `{category: colour}`. |
| `jitter` | How far to spread the dots sideways at random (`0` or `False` = in a straight line). |
| `alpha` | Transparency, from 0 (invisible) to 1 (solid). Useful when dots overlap. |
| `size` | Dot size. |
| `dodge` | With a `hue` that differs from `x`, `dodge=True` puts the hue groups side by side. |
| `ax` | Which axes to draw on (from `fig, ax = plt.subplots()`). |

**Related:** `sns.swarmplot()` takes the same arguments but arranges the dots so none overlap (good for small *n*,
slow for thousands of points).

### The same trials as a distribution

A histogram counts how many trials fall into each range ("bin") of reaction times. `kde=True` adds a smooth curve
over the bars: a **kernel density estimate (KDE)**, which is a smoothed outline of the histogram's shape.
Keep the KDE in mind; it comes back in a moment as the violin plot.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(
    data=one_person_df,
    x="rt_ms",
    hue="condition",
    palette=condition_colours,
    binwidth=25,        # each bar covers 25 ms
    element="step",     # draw outlines, so the overlapping distributions stay visible
    kde=True,           # add the smooth KDE curve on top of each histogram
    ax=ax,
)
ax.set(xlabel="Reaction time (ms)", ylabel="Number of trials", title="Participant 1")
plt.show()

Most trials pile up around 400 ms, with a long tail of occasional slow responses to the right. This shape is
typical of reaction times. The incongruent curve is shifted a little to the right of the other two.

#### Reference: `sns.histplot()`

```python
sns.histplot(data=df, x="number_column", hue="category_column", binwidth=25, element="step", kde=True, ax=ax)
```

| Argument | What it does |
|---|---|
| `x` | The numeric column to count. |
| `binwidth` or `bins` | Width of each bar, or the number of bars. Try a few: too wide hides the shape, too narrow looks noisy. |
| `hue` | Draw a separate histogram for each group, in different colours. |
| `element` | `"bars"` (default), `"step"` (outline only) or `"poly"`. Outlines are easier to read when groups overlap. |
| `stat` | What the y-axis shows: `"count"` (default), `"probability"` or `"density"`. |
| `common_norm` | With `stat="density"` and a `hue`, `common_norm=False` scales each group separately, so groups with fewer trials can still be compared by shape. |
| `kde` | `True` adds a smoothed curve (kernel density estimate). |

`histplot` can do a lot more than we use here: stacked histograms, cumulative histograms, log scales and
two-dimensional histograms. We won't cover those; see the seaborn documentation if you need them.
`sns.kdeplot()` draws only the smooth curve, without the bars.

### From the KDE to the violin plot

A **violin plot** draws the KDE curve for each group, turned on its side and mirrored, so it looks like a violin.
Here are the same trials again. Reaction time is now on the vertical axis, so tilt your head: the width of each
violin follows the height of the matching KDE curve above.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
# The same kind of arguments as the stripplot above; only the function name has changed.
# This time the conditions go on x and reaction time on y, so the violins stand upright.
# inner="quart" draws dashed lines at the median and the quartiles inside each violin.
# Sorting the rows by condition first avoids a seaborn 0.13 bug that can mix up the violin colours
# when the conditions appear in the data in a different order from condition_order.
sns.violinplot(
    data=one_person_df.sort_values("condition"),
    x="condition",
    y="rt_ms",
    hue="condition",
    palette=condition_colours,
    dodge=False,        # hue only colours the violins; don't shift them sideways
    # inner="quart",
    ax=ax,
)
ax.set(xlabel="", ylabel="Reaction time (ms)", title="Participant 1: KDE of trials, as violins")
plt.show()

The violin shows the same information as the KDE: where trials are common (wide) and rare (narrow).
What it no longer shows is the individual trials, or how many there were.

**Changing the view by changing the function name.** `stripplot`, `swarmplot`, `violinplot`, `boxplot`,
`barplot` and `pointplot` all take the same main arguments (`data`, `x`, `y`, `hue`, `palette`, `ax`).
To see your data a different way, you can usually just change the function name.

## Step 2: one point = one participant

Now summarise each person's trials into **one number per condition**: their mean reaction time.
After this step each row stands for about 200 trials.

In [ ]:
# One row per participant per condition: the mean of that person's correct trials.
participant_means_df = (
    correct_df
    .groupby(["participant", "condition"], as_index=False, observed=True)
    # Named aggregation: new_column_name=(column_to_summarise, function).
    # This makes two new columns at once: each person's mean RT and their number of trials.
    .agg(mean_rt_ms=("rt_ms", "mean"), n_trials=("rt_ms", "size"))
)
# The whole calculation is wrapped in ( ) so it can be split over several lines, one step per line.
participant_means_df.head(6)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
# Each dot is now a person.
sns.stripplot(
    data=participant_means_df,
    x="condition",
    y="mean_rt_ms",
    hue="condition",
    palette=condition_colours,
    jitter=0.15,
    size=5,
    ax=ax,
)
ax.set(xlabel="", ylabel="Mean reaction time (ms)", title="One dot = one participant")
plt.show()

Now we can count the people (47 dots per condition). The incongruent cloud is a bit higher, but the three clouds
**overlap a lot**: some people are slower on *congruent* trials than others are on *incongruent* trials.

Does that mean the effect is weak? Not necessarily. Everyone did every condition, so it is a
**within-participant** design, and the graph above throws that away: we can't tell which dot belongs to whom.
Joining each person's dots with a line puts it back.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

# By default lineplot AVERAGES all rows at each x value and draws one line.
# units="participant" says "draw a separate line for each participant" instead,
# and estimator=None turns the averaging off. You need both together.
sns.lineplot(
    data=participant_means_df,
    x="condition",
    y="mean_rt_ms",
    units="participant",
    estimator=None,
    color="0.8",       # a light grey, so the lines sit behind the dots
    linewidth=0.8,
    ax=ax,
)
# Draw the dots on top. jitter=False keeps each dot on its line.
sns.stripplot(
    data=participant_means_df,
    x="condition",
    y="mean_rt_ms",
    hue="condition",
    palette=condition_colours,
    jitter=False,
    size=5,
    ax=ax,
)
ax.set(xlabel="", ylabel="Mean reaction time (ms)", title="One line = one participant")
plt.show()

Nearly every line goes **up** from congruent (or neutral) to incongruent. The overlap in the previous graph
came from people differing in their **overall speed** (some lines are high, some low), not from the effect being
inconsistent.

> A within-participant comparison is about the **slope of each line**, not the overlap of the clouds.

#### Reference: `sns.lineplot()`

Draws lines through points. Last week we used it for one line per group over time. Two new arguments let it draw
one line per *person* (or subject, or neuron).

```python
sns.lineplot(data=df, x="time_or_condition", y="value", hue="group", units="participant", estimator=None, ax=ax)
```

| Argument | What it does |
|---|---|
| `x`, `y` | Usually `x` is time or condition and `y` is the measurement. |
| `hue`, `style` | Separate lines by colour and/or line style (solid, dashed...). |
| `estimator` | How to combine rows with the same `x`: the default is the mean. `estimator=None` means don't combine. |
| `units` | A column that identifies which rows belong to the same line, e.g. `"participant"`. Use with `estimator=None`. |
| `errorbar` | When averaging, the band around the line: default `("ci", 95)`. `None` removes it. See step 4. |
| `marker` | Add a marker at each point, e.g. `marker="o"`. |
| `color`, `linewidth`, `alpha` | One fixed colour (e.g. `"0.7"` is a grey: `"0"` is black, `"1"` is white), line thickness and transparency. |

## Step 3: one point = one participant's effect

If the question is "how big is the flanker effect?", we can plot it directly. For each person:

**flanker effect = mean incongruent RT − mean congruent RT**

`participant_means_df` has one row per participant *per condition*, so each person's congruent and incongruent
means are on different rows. To subtract them, we need them side by side on **one row per participant**.
We can do that with tools we already know, in two steps:

1. Make a `congruent` column that holds the mean RT on congruent rows and is empty on every other row,
   and an `incongruent` column in the same way.
2. `groupby("participant")` collects each person's rows into one. Averaging a column ignores the empty values,
   so each person's `congruent` value is simply their one congruent mean.

In [ ]:
# Work on a copy, so participant_means_df itself is not changed.
effect_calc_df = participant_means_df.copy()

# Step 1: Boolean masks (True/False for each row), as in week 3.
is_congruent = effect_calc_df["condition"] == "congruent"
is_incongruent = effect_calc_df["condition"] == "incongruent"

# .loc[rows, "new_column"] = values creates a new column and fills it only on the selected rows.
# All other rows are left empty: pandas marks them NaN ("not a number"), meaning missing.
effect_calc_df.loc[is_congruent, "congruent"] = effect_calc_df.loc[is_congruent, "mean_rt_ms"]
effect_calc_df.loc[is_incongruent, "incongruent"] = effect_calc_df.loc[is_incongruent, "mean_rt_ms"]

# Look at one participant's three rows: each new column has one value and two NaNs.
effect_calc_df.head(3)

In [ ]:
# Step 2: one row per participant.
# "mean" skips NaN values, and each person has exactly one non-missing value in each column,
# so the "mean" is just that value. groupby is collecting each person's values onto one row.
effect_df = (
    effect_calc_df
    .groupby("participant", as_index=False)
    .agg(congruent=("congruent", "mean"), incongruent=("incongruent", "mean"))
)

# The effect for each person: how much slower they are when the flankers conflict.
# Subtracting two columns subtracts them row by row, so each participant gets their own effect.
effect_df["flanker_effect_ms"] = effect_df["incongruent"] - effect_df["congruent"]
effect_df.head()

**Another tool you'll meet: `pivot()`.** Reshaping a table from *long* format (one row per participant per
condition) to *wide* format (one row per participant, one column per condition) is so common that pandas has a
function for it. The commented-out code below does the same job as the two cells above. We haven't covered it and
you don't need it for this module, but you will see it in other people's code and in AI-generated code.
To try it, remove the `# ` at the start of each line.

In [ ]:
# # pivot(index=what becomes the rows, columns=what becomes the new columns, values=what fills the cells)
# wide_df = participant_means_df.pivot(index="participant", columns="condition", values="mean_rt_ms")
#
# # wide_df now has one row per participant and the columns congruent, neutral and incongruent.
# wide_df["flanker_effect_ms"] = wide_df["incongruent"] - wide_df["congruent"]
#
# # pivot puts participant in the index (the row labels); reset_index() turns it back into a column.
# effect_df = wide_df.reset_index()
# effect_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 4))

# One dot per participant's effect. With only y= given, seaborn draws a single strip.
# color= (one colour for everything) is used instead of hue/palette because there is only one group.
sns.stripplot(data=effect_df, y="flanker_effect_ms", color=condition_colours["incongruent"],
              alpha=0.6, size=5, jitter=0.15, ax=ax)
# Mean and 95% confidence interval of the effect, drawn in black on top.
# markers="D" draws a diamond. Drawing it after the stripplot puts it on top.
sns.pointplot(data=effect_df, y="flanker_effect_ms", errorbar=("ci", 95),
              color="black", markers="D", ax=ax)
# A reference line at zero: "no effect".
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

ax.set(ylabel="Flanker effect (ms)\nincongruent − congruent", title="One dot = one person's effect")
plt.show()

In [ ]:
# Count how many people show a positive effect.
# (effect > 0) gives True/False for each person; .sum() counts the Trues.
n_positive = (effect_df["flanker_effect_ms"] > 0).sum()
print(f"{n_positive} of {len(effect_df)} participants were slower on incongruent trials")
effect_df["flanker_effect_ms"].describe().round(1)

Now the answer is easy to see: **every one of the 47 participants** shows the effect. The average is about
45 ms and individuals range from about 12 ms to over 120 ms. The 95% CI for the mean is tiny and far from zero.

This graph answers the scientific question better than the bar chart we started with, and it shows the raw
material (the people) at the same time.

If you know the paired t-test: it is a one-sample test on exactly these difference scores. So the graph that
matches a paired t-test is this one, not two bars.

#### Reference: `sns.pointplot()`

Draws the **mean** (or another summary) of each group as a point, with an error bar. It is a lighter alternative to
`barplot` and sits nicely on top of a `stripplot`.

```python
sns.pointplot(data=df, x="condition", y="value", errorbar=("ci", 95), color="black",
              markers="D", linestyle="none", capsize=0.2, ax=ax)
```

| Argument | What it does |
|---|---|
| `x`, `y` | As for the other categorical plots. Give only `y` for a single group. |
| `errorbar` | What the error bar shows (see step 4). |
| `markers` | Marker shape: `"o"` circle, `"D"` diamond, `"s"` square. |
| `linestyle` | By default pointplot joins the means with a line; `linestyle="none"` removes it. |
| `capsize` | Width of the horizontal caps at the ends of the error bars. |
| `color` | One colour for every point. Use `hue`/`palette` to colour by group instead. |

## Step 4: one point = the group mean, and what is its error bar?

A group mean is one number that summarises many people. The error bar around it can mean very different things,
and a reader can't tell which unless you say so. The three most common:

| Error bar | Answers the question... | More participants make it... |
|---|---|---|
| **SD** (standard deviation) | How much do *individuals* vary? | about the same |
| **SE** (standard error of the mean) | How precisely do we know the *mean*? | smaller |
| **95% CI** (confidence interval) | A range of plausible values for the *mean* (≈ mean ± 2 SE) | smaller |

Seaborn controls this with the `errorbar=` argument. Here is the **same data** drawn three ways.

#### Reference: the `errorbar=` argument

Works the same way in `barplot`, `pointplot`, `lineplot`, `catplot` and `relplot`.

| Value | Error bar shows |
|---|---|
| `"sd"` | mean ± 1 standard deviation |
| `"se"` | mean ± 1 standard error |
| `("ci", 95)` | 95% confidence interval for the mean (the default) |
| `("pi", 95)` | the range containing the middle 95% of the data points |
| `None` | no error bar |

Seaborn computes CIs by *bootstrapping* (resampling the data many times), so they can change very slightly each
time you run the cell. Add `seed=0` if you need identical results every time.

In [ ]:
# Three panels side by side; sharey=True puts them on one scale so they can be compared.
# axes holds the three panels: axes[0] is the left one, axes[1] the middle, axes[2] the right.
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=True)

# We draw the same plot three times. Only errorbar= and the title change.

# --- Left panel: standard deviation --------------------------------------------
# Show the people faintly in the background...
sns.stripplot(data=participant_means_df, x="condition", y="mean_rt_ms", hue="condition",
              palette=condition_colours, alpha=0.3, jitter=0.15, ax=axes[0])
# ...and the group mean with its error bar on top.
sns.pointplot(data=participant_means_df, x="condition", y="mean_rt_ms", errorbar="sd",
              color="black", linestyle="none", capsize=0.2, ax=axes[0])
axes[0].set(title="Error bars: standard deviation", xlabel="", ylabel="Mean reaction time (ms)")

# --- Middle panel: standard error -----------------------------------------------
sns.stripplot(data=participant_means_df, x="condition", y="mean_rt_ms", hue="condition",
              palette=condition_colours, alpha=0.3, jitter=0.15, ax=axes[1])
sns.pointplot(data=participant_means_df, x="condition", y="mean_rt_ms", errorbar="se",
              color="black", linestyle="none", capsize=0.2, ax=axes[1])
axes[1].set(title="Error bars: standard error", xlabel="")

# --- Right panel: 95% confidence interval ----------------------------------------
sns.stripplot(data=participant_means_df, x="condition", y="mean_rt_ms", hue="condition",
              palette=condition_colours, alpha=0.3, jitter=0.15, ax=axes[2])
sns.pointplot(data=participant_means_df, x="condition", y="mean_rt_ms", errorbar=("ci", 95),
              color="black", linestyle="none", capsize=0.2, ax=axes[2])
axes[2].set(title="Error bars: 95% confidence interval", xlabel="")

plt.show()

Three important points:

1. **The data are identical in all three panels.** Only the error bars changed. If a figure doesn't say which one
   it shows, the reader can't interpret it. Always state it in the figure legend.
2. **SD bars overlap heavily** here, yet we just saw that all 47 people show the effect. Overlapping error bars
   between conditions say nothing about a **within-participant** effect, because they include the
   differences in overall speed between people. To show a within-participant effect, show the differences (step 3).
3. SE and CI describe how precisely the *mean* is known. They are about **the group**, not about how much
   individuals vary.

### Back to the opening figure: what did its error bar average over?

The opening bar chart was made from `correct_df`, in which **one row is one trial**. Seaborn treated around
30,000 trials as 30,000 independent observations, but they come from only 47 people. Compare it with the same
bar chart made from the participant means.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)

# axes[0] is the left panel, axes[1] the right one.
# Left: one row = one trial (what we did at the start).
sns.barplot(data=correct_df, x="condition", y="rt_ms", hue="condition",
            palette=condition_colours, dodge=False, errorbar=("ci", 95), capsize=0.2, ax=axes[0])
axes[0].set(title="Averaging trials\n(n ≈ 30,000 'observations')", xlabel="",
            ylabel="Mean reaction time (ms)")

# Right: one row = one participant (the correct unit for a claim about people).
sns.barplot(data=participant_means_df, x="condition", y="mean_rt_ms", hue="condition",
            palette=condition_colours, dodge=False, errorbar=("ci", 95), capsize=0.2, ax=axes[1])
axes[1].set(title="Averaging participants\n(n = 47)", xlabel="")

plt.show()

The bar heights are almost the same, but the left-hand CIs are much too narrow: the graph claims far more
precision than 47 people can give. **Before plotting, decide what one row should be.** For claims about people,
summarise to one value per person per condition first.

### Bars hide the data; dots show it

A bar chart of means shows only one number per condition. The **dots + mean ± CI** plot from step 4 shows the same
mean and also the people behind it, so it is a better default for psychology data with tens of participants.
When there are too many points to draw, show the *shape* of the distribution instead, for example with a violin
plot or a box plot. Same data and same arguments; only the function name changes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)

# A violin is a KDE (smoothed histogram), mirrored. inner="quart" marks the median and quartiles.
sns.violinplot(data=participant_means_df, x="condition", y="mean_rt_ms", hue="condition",
               palette=condition_colours, dodge=False, inner="quart", ax=axes[0])
axes[0].set(title="Violin plot", xlabel="", ylabel="Mean reaction time (ms)")

# A box plot: the box spans the middle 50% of people, the line is the median.
sns.boxplot(data=participant_means_df, x="condition", y="mean_rt_ms", hue="condition",
            palette=condition_colours, dodge=False, width=0.5, ax=axes[1])
axes[1].set(title="Box plot", xlabel="")

plt.show()

In these two graphs, one data point is still one participant, but the individual people are no longer drawn.
Both shapes summarise the 47 dots. In step 1 the violin summarised one person's trials; here it summarises
people's means. **Always check which one a violin is showing.**

You may see *raincloud plots* in papers (Allen et al., 2019). They combine a half-violin, a box plot and the
dots, to show the shape and the individual data points together.

| Plot | Shows | Use when |
|---|---|---|
| `stripplot` / `swarmplot` | every data point | up to ~100 points per group |
| `boxplot` | median, quartiles and outliers | many points; you want a compact summary |
| `violinplot` | the shape of the distribution (KDE) | many points; the shape matters (skew, two peaks) |
| `pointplot` / `barplot` | one summary (mean) ± error bar | combined with dots, or for counts and proportions |

### Figure-level plots: `catplot` makes the panels for you

So far we've made a figure with `plt.subplots()` and passed `ax=` to each seaborn function. Seaborn can also build
the whole figure, including several panels, in one call. `sns.catplot()` does this for all the categorical plots
above.

The trials were run in 5 blocks. Does the flanker effect survive practice? We need one mean per participant,
per block, per condition.

In [ ]:
# Group by three columns at once: one row per participant x block x condition.
block_means_df = (
    correct_df
    .groupby(["participant", "block", "condition"], as_index=False, observed=True)
    .agg(mean_rt_ms=("rt_ms", "mean"))
)
block_means_df.head()

In [ ]:
# kind= chooses the type of plot; col= makes one panel per block.
# catplot creates its own figure, so there is no fig, ax = plt.subplots() and no ax= argument.
# It returns a FacetGrid object, which we store as g so we can change the labels afterwards.
g = sns.catplot(
    data=block_means_df,
    x="condition", y="mean_rt_ms",
    hue="condition", palette=condition_colours,
    col="block",              # one panel (column) for each value of block
    kind="strip",             # try "violin", "box" or "point" instead
    height=3, aspect=0.7,     # each panel is 3 inches tall and 0.7 x 3 inches wide
)
g.set_axis_labels("", "Mean reaction time (ms)")     # x label, y label, applied to every panel
g.set_titles("Block {col_name}")                     # {col_name} is replaced by each panel's block number
g.tick_params(axis="x", labelrotation=45)           # tilt the x labels in every panel so they don't overlap
plt.show()

The same plot with `kind="point"` and the conditions as coloured lines makes the comparison across blocks easier:

In [ ]:
# Here block goes on the x-axis and condition becomes the hue, so there is just one panel.
g = sns.catplot(
    data=block_means_df,
    x="block", y="mean_rt_ms",
    hue="condition", palette=condition_colours,
    kind="point",
    errorbar=("ci", 95),
    dodge=0.3,                # shift the hue groups sideways a little so their error bars don't overlap
    height=4, aspect=1.3,
)
g.set_axis_labels("Block", "Mean reaction time (ms)")
plt.show()

Mean reaction times change little across the blocks, and incongruent trials are slower than the other two
conditions in every block. The gap shrinks a little in the last block, but the flanker effect does not go away
with practice. Each point here is the mean of 47 participants' block means.

### Rows *and* columns

`col=` and `row=` can be used together to make a grid of panels: one variable across, another down.
On each trial the centre arrow pointed either left or right (`arrow_direction`). Here each **row** is one arrow
direction and each **column** is one block, so every panel shows one direction in one block.

In [ ]:
# One mean per participant x arrow direction x block x condition (four grouping columns this time).
direction_means_df = (
    correct_df
    .groupby(["participant", "arrow_direction", "block", "condition"], as_index=False, observed=True)
    .agg(mean_rt_ms=("rt_ms", "mean"))
)
direction_means_df.head()

In [ ]:
g = sns.catplot(
    data=direction_means_df,
    x="condition", y="mean_rt_ms",
    hue="condition", palette=condition_colours,
    row="arrow_direction",    # one row of panels for each arrow direction (left, right)
    col="block",              # one column of panels for each block (1 to 5)
    kind="point",             # mean ± 95% CI in each panel; try "strip" or "box" instead
    height=2.5, aspect=0.8,   # height and aspect set the size of EACH panel; the grid is 5 panels wide, 2 tall
)
g.set_axis_labels("", "Mean RT (ms)")
# With rows and columns, {row_name} and {col_name} are both available for the titles.
g.set_titles("{row_name} arrow, block {col_name}")
g.tick_params(axis="x", labelrotation=45)
plt.show()

Ten panels from one call. Every panel shares the same y-axis, so they can be compared directly.

In this layout the three conditions sit **side by side within each panel**, so it's easy to compare conditions,
which is the question the experiment was designed to answer. But suppose the question is *"do left and right
arrows give different reaction times?"* Left and right are now in **different rows**, far apart, so the reader has
to compare dots across panels by eye.

### Put what you want to compare next to each other

The closer two things are on a figure, the easier they are to compare. A useful rule for choosing the roles:

1. **x-axis**: the comparison you care about most (side by side, same panel)
2. **hue**: the next most important (different colours, same panel)
3. **col**: next (neighbouring panels, same height)
4. **row**: least important (panels above and below)

To compare left with right, we give `arrow_direction` the x-axis and move `condition` to the rows.
The data are exactly the same; only the roles change.

In [ ]:
g = sns.catplot(
    data=direction_means_df,
    x="arrow_direction", y="mean_rt_ms",      # left and right now sit side by side in each panel
    hue="condition", palette=condition_colours,   # keep the condition colours, so each row has its colour
    row="condition",          # one row of panels per condition
    col="block",              # one column of panels per block, as before
    kind="point",
    height=2.2, aspect=0.9,
)
g.set_axis_labels("Arrow direction", "Mean RT (ms)")
g.set_titles("{row_name}, block {col_name}")
plt.show()

Now each panel puts left next to right, and it's easy to see they are almost identical: arrow direction makes no
difference. Comparing the conditions is harder in this version, because they are in different rows.

Neither layout is "correct". **Decide which comparison your figure is for, and put those things next to each other.**

#### Reference: `sns.catplot()` and figure-level plots

Seaborn has two kinds of plotting functions:

| | **Axes-level** (e.g. `stripplot`, `histplot`, `lineplot`) | **Figure-level** (`catplot`, `displot`, `relplot`) |
|---|---|---|
| Draws on | one axes that you made: `ax=ax` | a new figure that it makes itself |
| Several panels | you make them with `plt.subplots()` | `col=` and `row=` make them for you |
| Figure size | `plt.subplots(figsize=(width, height))` | `height=` (of each panel) and `aspect=` (width ÷ height) |
| Returns | an axes | a `FacetGrid`, stored as `g` |
| Change labels with | `ax.set(...)` | `g.set_axis_labels(...)`, `g.set_titles(...)`, `g.set(...)` |

```python
g = sns.catplot(data=df, x="condition", y="value", hue="group", col="block", row="session",
                kind="strip", height=3, aspect=1)
```

| Argument | What it does |
|---|---|
| `kind` | `"strip"`, `"swarm"`, `"box"`, `"violin"`, `"boxen"`, `"point"`, `"bar"` or `"count"`. The same as calling the axes-level function with that name. |
| `col`, `row` | A column whose values each get their own column or row of panels. |
| `col_wrap` | Start a new row after this many panels (use with `col`, not with `row`). |
| `height`, `aspect` | Height of each panel in inches, and its width as a multiple of the height. |
| `sharex`, `sharey` | Whether panels share axis scales (default `True`). |
| Any other argument | Passed on to the underlying plot, e.g. `errorbar=`, `jitter=`, `inner=`, `dodge=`. |

After plotting:

| Method | What it does |
|---|---|
| `g.set_axis_labels("x label", "y label")` | Label the axes of every panel. |
| `g.set_titles("Block {col_name}")` | Panel titles; `{col_name}` and `{row_name}` are filled in for you. |
| `g.set(ylim=(0, 600))` | Set any axes property on every panel. |
| `g.refline(x=0)` or `g.refline(y=0)` | Draw a reference line in every panel. Takes the usual line options, e.g. `color="black", linestyle=":"`. |
| `g.tick_params(axis="x", labelrotation=45)` | Change the tick labels in every panel, e.g. tilt them. |
| `g.savefig("file.pdf")` | Save the whole figure. |

The three figure-level functions match the axes-level ones: `catplot` ↔ strip/swarm/box/violin/point/bar plots;
`displot` ↔ `histplot`, `kdeplot`, `ecdfplot`; `relplot` ↔ `scatterplot`, `lineplot`.

## Step 5: one point = the group's average signal at one moment

Neuroscience data often unfold over **time**. These fMRI data come from Waskom et al. (2017), *Cerebral Cortex*
(they are also used as an example dataset in seaborn).

In [ ]:
# Each row: one subject's BOLD signal in one brain region, at one time step after one type of event.
fmri_df = pd.read_csv("data/fmri_waskom2017.csv")
print("Subjects:", fmri_df["subject"].nunique())
fmri_df.head()

**What does one row show?** One subject's average BOLD signal in one region (`parietal` or `frontal`),
`timepoint` steps after an event (`stim` or `cue`). The summarising began before the data reached us: each value is
already an average over that subject's trials and over the voxels in the region. Some of the ladder has
already been climbed, and we can't go back down.

First, the group average over time for the parietal region:

In [ ]:
# Select the parietal region only (a Boolean mask inside .loc, as in week 3).
parietal_df = fmri_df.loc[fmri_df["region"] == "parietal"]

fig, ax = plt.subplots(figsize=(6, 4))
# lineplot averages the subjects at each timepoint and draws a 95% CI band by default.
sns.lineplot(data=parietal_df, x="timepoint", y="signal", hue="event", ax=ax)
ax.axhline(0, color="black", linewidth=0.6)
ax.set(xlabel="Time after event (scans)", ylabel="BOLD signal",
       title="Parietal cortex (line = mean of 14 subjects, band = 95% CI)")
plt.show()

How to read it:

- Each point on a line is **the mean of 14 subjects at one time step**.
- The band is a **95% CI for that mean**. It is not the range of the subjects.
- The signal peaks several scans *after* the event, because the blood-flow response the scanner measures is slow
  (the haemodynamic response).
- The response is larger after the stimulus than after the cue.

To check what the band hides, draw each subject separately:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
# units + estimator=None: one line per subject, no averaging.
sns.lineplot(data=parietal_df, x="timepoint", y="signal", hue="event",
             units="subject", estimator=None, linewidth=0.7, alpha=0.6, ax=ax)
ax.axhline(0, color="black", linewidth=0.6)
ax.set(xlabel="Time after event (scans)", ylabel="BOLD signal",
       title="Parietal cortex (one line = one subject)")
plt.show()

Individual subjects spread much more widely than the CI band. Both graphs are correct; they answer different
questions ("what is the average response?" versus "how much do people vary?").

### Small multiples: one panel per region

To compare regions, `relplot` (a *figure-level* function) puts each region in its own panel, called a *facet*.
`col="region"` gives one column per region.

In [ ]:
# relplot is the figure-level version of lineplot/scatterplot (like catplot for categorical plots).
# kind="line" uses lineplot inside each panel; col="region" makes one panel per region.
g = sns.relplot(data=fmri_df, x="timepoint", y="signal", hue="event", col="region",
                kind="line", height=3.5, aspect=1.1)
g.set_axis_labels("Time after event (scans)", "BOLD signal")
g.set_titles("{col_name} cortex")   # {col_name} is filled in with each region's name
plt.show()

By default all panels **share the same y-axis**, so we can see the frontal response is smaller.

**Chart crime: unshared axes.** Watch what happens if each panel gets its own scale:

In [ ]:
# facet_kws passes options to the FacetGrid; sharey=False gives each panel its own y-axis.
g = sns.relplot(data=fmri_df, x="timepoint", y="signal", hue="event", col="region",
                kind="line", height=3.5, aspect=1.1, facet_kws={"sharey": False})
g.set_axis_labels("Time after event (scans)", "BOLD signal")
g.set_titles("{col_name} cortex")
plt.show()

Now the frontal response *looks* as big as the parietal one. Readers compare panels by eye, so if panels are
meant to be compared, **give them the same axes**. If you have a good reason not to, say so in the legend.

#### Reference: `sns.relplot()`

The figure-level function for **relationships between two numeric variables**: scatter plots and line plots,
optionally split into panels. It works like `catplot` (see the figure-level reference in step 4): it makes its own
figure, returns a `FacetGrid` `g`, and is sized with `height` and `aspect`.

```python
g = sns.relplot(data=df, x="time", y="value", hue="event", style="choice", col="region", row="group",
                kind="line", height=3.5, aspect=1.1, facet_kws={"sharey": True})
```

| Argument | What it does |
|---|---|
| `kind` | `"scatter"` (default) or `"line"`. |
| `x`, `y` | Two numeric columns. |
| `hue` | Colour by a column. A *numeric* hue gets a graded (sequential) palette automatically. |
| `style` | Line style (solid, dashed) or marker shape by a column. |
| `size` | Marker size or line width by a column. |
| `col`, `row`, `col_wrap` | Panels, as in `catplot`. |
| `units`, `estimator`, `errorbar` | As in `lineplot` (`kind="line"` only). |
| `facet_kws` | A dictionary of options for the panel grid itself, e.g. `{"sharey": False}` or `{"sharex": False}`. Use sparingly. |

## Step 6: many conditions at once: neurons making a decision

Roitman & Shadlen (2002) recorded neurons in the lateral intraparietal area (LIP) of monkeys. The monkeys watched
a cloud of moving dots and decided which way the dots were moving, then reported it with an eye movement
(saccade) to one of two targets. `coherence_pct` is the percentage of dots moving together, so higher means easier.
`T1` is the target inside the neurons' response field.

In [ ]:
lip_df = pd.read_csv("data/lip_roitman_shadlen2002.csv")
lip_df.head()

**What does one row show?** The average firing rate (spikes per second) of the recorded LIP neurons, averaged over
many neurons and many trials, at one moment, for one motion strength and one choice. Time is measured either from
**motion onset** or from the **saccade**. We are near the top of the ladder: we can't see single neurons or
single trials.

There are four variables to show: time, coherence, choice and alignment. Each gets a job:

- `x` = time, `y` = firing rate
- `hue` = coherence. It is **ordered** (0 → 51.2%), so it gets an **ordered (sequential) palette**, from light to dark.
- `style` = choice (solid or dashed line)
- `col` = alignment (one panel each)

In [ ]:
g = sns.relplot(
    data=lip_df, kind="line",
    x="time_ms", y="firing_rate_hz",
    hue="coherence_pct", palette="flare",       # sequential palette for an ordered variable
    style="choice",
    col="align",
    facet_kws={"sharex": False},                 # the two alignments cover different time ranges,
                                                 # but the y-axis (firing rate) is still shared
    height=3.8, aspect=1.1,
)
g.set_axis_labels("Time (ms)", "Firing rate (spikes/s)")
g.set_titles("Aligned to {col_name}")
# Mark time zero (motion onset, or the moment of the saccade) in every panel.
# g.refline(x=0) draws a vertical reference line at x = 0 in each panel of the grid.
g.refline(x=0, color="black", linewidth=0.6, linestyle=":")
plt.show()

How to read it:

- **Left (aligned to motion onset):** for choices into the response field (solid lines), firing rate ramps up.
  The stronger the motion, the faster the ramp.
- **Right (aligned to the saccade):** the solid lines converge on a similar level just before the eye
  movement, whatever the motion strength. This is the evidence for a **decision threshold**: the neurons
  accumulate evidence until a fixed level is reached, then the monkey responds.

**Chart crime: rainbow colours for ordered data.** If coherence were coloured with an unordered palette
(red, green, blue...), the reader would have to look up every colour in the legend. A sequential palette lets them
see "darker = stronger motion" at a glance. Rainbow colour maps (such as *jet*) also create false boundaries and
fail for colour-blind readers.

#### Reference: choosing a palette

| The variable is... | Use a palette that is... | Seaborn examples |
|---|---|---|
| unordered categories (conditions, groups) | qualitative: clearly different colours | `"colorblind"`, or your own dict of colours |
| ordered or numeric (coherence, dose, age) | sequential: light → dark | `"flare"`, `"crest"`, `"rocket"`, `"mako"`, `"viridis"` |
| above/below a meaningful middle (change from 0) | diverging: two colours meeting at a neutral middle | `"vlag"`, `"icefire"`, `"coolwarm"` |

Pass any of these as `palette=`. `sns.color_palette("flare", 6)` shows you a palette's colours.

## Step 7: building a figure for a paper

A paper figure tells **one story in several panels**. Ours: *every participant shows a flanker effect.*

- **A**: one participant's trials (the raw measurement)
- **B**: each participant's condition means, joined (the within-participant design)
- **C**: each participant's effect, with the group mean and 95% CI (the answer)

Things journals expect:

- **Size**: a double-column figure is about 180 mm wide (single column is about 85 mm). Matplotlib uses inches, so divide mm by 25.4.
- **Readable text** at that size: `sns.set_context("paper")` scales fonts down for print.
- **Panel letters**, so the legend and text can refer to "Fig. 1B".
- **The same colours mean the same thing** in every panel.
- **Vector output** (PDF or SVG) stays sharp at any zoom. Use PNG at 300 dpi or more if you need an image.

In [ ]:
# Paper context: smaller fonts and lines, sized for print.
sns.set_context("paper", font_scale=1.25)

mm = 1 / 25.4   # inches per millimetre
# subplot_mosaic lays out panels by name; each letter in the string is one panel.
# "ABC" = three panels in a row. It returns a dictionary: axd["A"] is panel A's axes, and so on.
# width_ratios sets the relative widths; layout="constrained" stops labels overlapping.

mosaic = "ABC"
width_ratios=[1.4, 1.2, 0.6]

mosaic = """
    AA
    BC
    """
width_ratios=[.4,.2]


fig, axd = plt.subplot_mosaic(mosaic, figsize=(180 * mm, 100 * mm),
                              width_ratios=width_ratios, layout="constrained")

# --- A: one participant, one dot = one trial ----------------------------------
sns.stripplot(data=one_person_df, x="rt_ms", y="condition", hue="condition",
              palette=condition_colours, jitter=0.3, alpha=0.4, size=2, ax=axd["A"])
axd["A"].set(xlabel="Reaction time (ms)", ylabel="", title="Participant 1: trials")

# --- B: one line = one participant ----------------------------------------------
sns.lineplot(data=participant_means_df, x="condition", y="mean_rt_ms", units="participant",
             estimator=None, color="0.75", linewidth=0.6, ax=axd["B"])
sns.stripplot(data=participant_means_df, x="condition", y="mean_rt_ms", hue="condition",
              palette=condition_colours, jitter=False, size=3, ax=axd["B"])
axd["B"].set(xlabel="", ylabel="Mean reaction time (ms)", title="All participants (n = 47)")
axd["B"].tick_params(axis="x", labelrotation=20)

# --- C: one dot = one participant's effect ---------------------------------------
sns.stripplot(data=effect_df, y="flanker_effect_ms", color=condition_colours["incongruent"],
              alpha=0.5, size=3, jitter=0.2, ax=axd["C"])
sns.pointplot(data=effect_df, y="flanker_effect_ms", errorbar=("ci", 95), color="black",
              markers="D", markersize=4, ax=axd["C"])
axd["C"].axhline(0, color="black", linewidth=0.6, linestyle="--")
axd["C"].set(ylabel="Flanker effect (ms)", title="Effect")

# Remove the top and right axis lines (a common journal style).
sns.despine(fig=fig)

# Add a bold panel letter at the top-left of each panel.
# ax.text(x, y, text) writes text. transform=ax.transAxes means x and y are fractions of the panel,
# where (0, 0) is the bottom-left and (1, 1) the top-right, so (-0.02, 1.02) is just above the top-left corner.
# ha and va are the horizontal and vertical alignment of the text.
axd["A"].text(-0.02, 1.02, "A", transform=axd["A"].transAxes, fontsize=12, fontweight="bold", ha="right", va="bottom")
axd["B"].text(-0.02, 1.02, "B", transform=axd["B"].transAxes, fontsize=12, fontweight="bold", ha="right", va="bottom")
axd["C"].text(-0.02, 1.02, "C", transform=axd["C"].transAxes, fontsize=12, fontweight="bold", ha="right", va="bottom")

# Save as vector PDF and PNG (for slides). bbox_inches="tight" trims white space.
# fig.savefig("figures/figure1_flanker.pdf", bbox_inches="tight")
# fig.savefig("figures/figure1_flanker.png", dpi=300, bbox_inches="tight")
plt.show()

# Return to the default context for anything that follows.
sns.set_context("notebook")

#### Reference: building a multi-panel figure

| Command | What it does |
|---|---|
| `sns.set_context("paper")` | Scales fonts and lines for print. Others: `"notebook"` (default), `"talk"`, `"poster"`. `font_scale=1.1` adjusts further. |
| `plt.subplot_mosaic("AB", figsize=...)` | Named panels in a row. `"AB;CC"` gives A and B on top and one wide C below. Returns `fig, axd`; use `axd["A"]`. |
| `width_ratios=[2, 1]`, `height_ratios=[...]` | Relative sizes of columns and rows. |
| `layout="constrained"` | Automatically spaces panels so labels don't overlap. |
| `sns.despine(fig=fig)` | Removes the top and right axis lines. |
| `ax.text(x, y, "A", transform=ax.transAxes)` | Writes a panel letter; `transAxes` means positions are fractions of the panel. |
| `ax.axhline(y)`, `ax.axvline(x)` | Horizontal or vertical reference line (e.g. at zero). |
| `fig.savefig("figures/name.pdf", bbox_inches="tight")` | Saves the figure. `.pdf` and `.svg` are vector formats; for `.png` add `dpi=300`. |

Size: 1 inch = 25.4 mm. Journals usually ask for ~85 mm (single column) or ~180 mm (double column) wide.
Figure-level plots (`catplot`, `relplot`) make their own figure, so they can't go into a mosaic panel; use the
axes-level functions there.

### The figure legend matters 
A reader should understand a figure from the figure and its legend alone. A good legend says **what one mark
represents**, **how many** there are, and **what the error bars are**:

> **Figure 1. Every participant showed a flanker effect.** **(A)** Reaction times on individual correct trials for
> one participant; each dot is one trial. **(B)** Mean reaction time on correct trials for each participant
> (n = 47) in each condition; lines join the same participant's means. **(C)** Flanker effect (incongruent minus
> congruent mean RT) for each participant (dots). The black diamond and bar show the group mean and its 95%
> confidence interval; the dashed line marks zero (no effect). Data from Hedge et al. (2018).

## Checklist: reading (and making) a figure

Use this for every figure you make, and for every figure you read in a paper. Next week you will use AI coding
tools to write plotting code, and this is how to judge what they give you.

1. **What does one mark represent?** A trial, a person, a group mean? Which measurement did it come from?
2. **How many** units (trials, people, neurons) are behind each mark? Can the reader see *n*?
3. **What do the error bars or bands show?** SD, SE or 95% CI? Is it stated?
4. **Within or between?** If the same people did every condition, does the graph show the pairing (lines) or the differences?
5. **Scales:** do panels that should be compared share their axes? Do bar charts start at zero?
6. **Colour:** does each colour mean the same thing everywhere? Ordered variables get ordered colours. Is it colour-blind safe?
7. **Legend:** could someone understand the figure from the figure and its legend alone?

## Command summary

| Command | One mark is... | Introduced in |
|---|---|---|
| `sns.stripplot()` / `sns.swarmplot()` | one row (a trial, a person...) | step 1 |
| `sns.histplot(..., kde=True)` | a count of rows in a bin, plus a smoothed curve | step 1 |
| `sns.violinplot()` / `sns.boxplot()` | the shape or quartiles of many rows | steps 1 and 4 |
| `sns.lineplot(..., units=, estimator=None)` | one line per unit (person, subject) | step 2 |
| `sns.pointplot(..., errorbar=)` | a group mean ± error bar | steps 3 and 4 |
| `sns.catplot(kind=..., col=)` | any categorical plot, split into panels | step 4 |
| `sns.relplot(kind="line", col=, style=)` | line or scatter plots, split into panels | steps 5 and 6 |
| `plt.subplot_mosaic()`, `sns.set_context("paper")`, `fig.savefig()` | building and saving a paper figure | step 7 |

### References

- Hedge, C., Powell, G., & Sumner, P. (2018). The reliability paradox: Why robust cognitive tasks do not produce reliable individual differences. *Behavior Research Methods, 50*, 1166–1186.
- Waskom, M. L., Frank, M. C., & Wagner, A. D. (2017). Adaptive engagement of cognitive control in context-dependent decision making. *Cerebral Cortex, 27*, 1270–1284.
- Roitman, J. D., & Shadlen, M. N. (2002). Response of neurons in the lateral intraparietal area during a combined visual discrimination reaction time task. *Journal of Neuroscience, 22*, 9475–9489.
- Weissgerber, T. L., Milic, N. M., Winham, S. J., & Garovic, V. D. (2015). Beyond bar and line graphs: Time for a new data presentation paradigm. *PLOS Biology, 13*, e1002128.
- Allen, M., Poggiali, D., Whitaker, K., Marshall, T. R., & Kievit, R. A. (2019). Raincloud plots: a multi-platform tool for robust data visualization. *Wellcome Open Research, 4*, 63.